# Gold tables for the dashboard and Genie
Deliverables 3.5 and 3.6. This notebook reads the cleaned listings (silver) and the model's predictions (gold) and builds the small set of gold tables that the AI/BI dashboard (`dashboard/airbnb_dashboard.lvdash.json`) and the Genie space (`genie/genie_setup.md`) sit on.

It runs over every city that is in silver, so city C shows up in the dashboard and in Genie as soon as the pipeline has loaded it. Run it as the last task of the job, after the model.

**Inputs** (full table names are widgets, so switching to the pipeline's tables is a widget change, not a code change):
- listings: one row per listing, all cities, with the columns in `LISTING_COLS` below. Until the ETL (3.3) writes `workspace.airbnb.silver_listings`, the default is the EDA table `workspace.airbnb_eda.listings_classified`, the same one the price model reads.
- predictions: the table written by `notebooks/price_model.ipynb` (`gold_price_predictions`). Used columns: `city`, `id`, `split`, `predicted_price`, `baseline_price`, `model_name`. `split` is `train` / `test` / `not_labelled` for A and B and `new_data` for city C, which the model never saw.

**Output** (schema `workspace.airbnb`): `gold_listings`, `gold_city_summary`, `gold_neighbourhood_stats`, `gold_model_errors`, `gold_model_metrics`.

In [ ]:
dbutils.widgets.text("listings_table", "workspace.airbnb_eda.listings_classified")
dbutils.widgets.text("predictions_table", "workspace.airbnb_eda.gold_price_predictions")
dbutils.widgets.text("output_schema", "workspace.airbnb")

SILVER = dbutils.widgets.get("listings_table")
PREDICTIONS = dbutils.widgets.get("predictions_table")
S = dbutils.widgets.get("output_schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")

# which city is A, B and C; Genie uses this to answer "City A" questions
CITY_ROLES = {"Los_Angeles": "A", "San_Francisco": "B", "Santa_Clara_County": "C"}

## 1. Check the inputs
Fail early with a readable message instead of a SQL error halfway through.

In [ ]:
LISTING_COLS = ["city", "id", "name", "neighbourhood_cleansed", "room_type", "property_type", "stay_type",
                "accommodates", "bedrooms", "price", "price_class", "is_stale",
                "availability_365", "review_scores_rating", "number_of_reviews", "host_id", "latitude", "longitude"]
PRED_COLS = ["city", "id", "split", "predicted_price", "baseline_price"]

listing_cols = spark.table(SILVER).columns
missing = sorted(set(LISTING_COLS) - set(listing_cols))
assert not missing, f"{SILVER} is missing {missing}. Rename them in the pipeline or adjust LISTING_COLS and the SQL below."

# silver is planned to carry price_valid; the EDA table only has price_class (same rule: priced and not a placeholder)
PRICE_VALID = "s.price_valid" if "price_valid" in listing_cols else "s.price_class NOT IN ('no_price', 'placeholder')"

HAS_PREDICTIONS = spark.catalog.tableExists(PREDICTIONS)
if HAS_PREDICTIONS:
    pred_cols = spark.table(PREDICTIONS).columns
    missing = sorted(set(PRED_COLS) - set(pred_cols))
    assert not missing, f"{PREDICTIONS} is missing {missing}."
    MODEL_RUN = "p.model_name" if "model_name" in pred_cols else "'price_model_v1'"
else:
    print(f"{PREDICTIONS} does not exist yet: the model tables are skipped, the dashboard's Model page will be empty.")

cities = [r.city for r in spark.table(SILVER).select("city").distinct().collect()]
print("cities in silver:", cities)
print("cities without a role (add them to CITY_ROLES):", [c for c in cities if c not in CITY_ROLES])

spark.createDataFrame(list(CITY_ROLES.items()), "city string, city_role string").createOrReplaceTempView("city_roles")

## 2. `gold_listings`
One row per listing, every city. Same rows as silver, only the columns the dashboard and Genie need, plus a readable city name and the city's role (A, B, C).

`price` here is only the **valid** price: placeholders ($99,999 hotel rooms) and missing prices become NULL, so any median or average over this column is safe. `price_class` still tells why a price is missing.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {S}.gold_listings AS
SELECT
  s.city,
  replace(s.city, '_', ' ')                        AS city_name,
  r.city_role,
  s.id,
  s.name,
  concat('https://www.airbnb.com/rooms/', s.id)    AS listing_url,
  s.neighbourhood_cleansed                         AS neighbourhood,
  s.room_type,
  s.property_type,
  s.stay_type,
  s.accommodates,
  s.bedrooms,
  CASE WHEN {PRICE_VALID} THEN s.price END         AS price,
  s.price_class,
  s.is_stale,
  s.availability_365,
  s.review_scores_rating,
  s.number_of_reviews,
  s.host_id,
  s.latitude,
  s.longitude
FROM {SILVER} s
LEFT JOIN city_roles r ON s.city = r.city
""")

## 3. `gold_city_summary`
One row per city. Prices are split by stay type: monthly rentals (`minimum_nights >= 28`) are quoted at about 60 % of a short stay per night (EDA, section 8), so one median over both would mislead.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {S}.gold_city_summary AS
SELECT
  city, city_name, city_role,
  count(*)                                                            AS listings,
  count(DISTINCT host_id)                                             AS hosts,
  round(count(*) / count(DISTINCT host_id), 2)                        AS listings_per_host,
  count(price)                                                        AS listings_with_valid_price,
  round(median(CASE WHEN stay_type = 'short' THEN price END), 2)      AS median_price_short_stay,
  round(median(CASE WHEN stay_type = 'monthly' THEN price END), 2)    AS median_price_monthly_stay,
  round(median(CASE WHEN stay_type = 'short' THEN price / accommodates END), 2) AS median_price_per_guest_short_stay,
  round(100 * avg(CASE WHEN room_type = 'Entire home/apt' THEN 1 ELSE 0 END), 1) AS pct_entire_home,
  round(100 * avg(CASE WHEN stay_type = 'monthly' THEN 1 ELSE 0 END), 1)        AS pct_monthly_rentals,
  round(100 * avg(CASE WHEN is_stale THEN 1 ELSE 0 END), 1)                     AS pct_stale,
  median(availability_365)                                            AS median_availability_365,
  round(100 * avg(CASE WHEN availability_365 = 0 THEN 1 ELSE 0 END), 1)         AS pct_zero_availability,
  round(avg(review_scores_rating), 2)                                 AS avg_review_score,
  round(100 * avg(CASE WHEN number_of_reviews = 0 THEN 1 ELSE 0 END), 1)        AS pct_without_reviews
FROM {S}.gold_listings
GROUP BY city, city_name, city_role
""")
display(spark.table(f"{S}.gold_city_summary").orderBy("city_role"))

## 4. `gold_neighbourhood_stats`
One row per (city, neighbourhood). Neighbourhood names are city-specific, so always filter by city when reading this table.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {S}.gold_neighbourhood_stats AS
SELECT
  city, city_name, city_role, neighbourhood,
  count(*)                                                            AS listings,
  count(CASE WHEN stay_type = 'short' THEN price END)                 AS short_stay_listings_with_price,
  round(median(CASE WHEN stay_type = 'short' THEN price END), 2)      AS median_price_short_stay,
  round(median(CASE WHEN stay_type = 'monthly' THEN price END), 2)    AS median_price_monthly_stay,
  round(100 * avg(CASE WHEN room_type = 'Entire home/apt' THEN 1 ELSE 0 END), 1) AS pct_entire_home,
  round(avg(review_scores_rating), 2)                                 AS avg_review_score,
  median(availability_365)                                            AS median_availability_365
FROM {S}.gold_listings
GROUP BY city, city_name, city_role, neighbourhood
""")

## 5. Model tables
`gold_model_errors`: one row per (listing, model run) **with a valid actual price**. `error = predicted - actual`, so a negative error means the model **under-prices** the listing.

`is_evaluation` marks the rows whose errors we report: the `test` split of A and B, and every priced listing of city C (`new_data`), since the model never saw C. `train` rows stay in the table for comparison only.

`gold_model_metrics`: MAE of the model and of the baseline per city, model run and split, plus one `A + B combined` row. The numbers to report are the rows with `is_evaluation = true`.

In [ ]:
if HAS_PREDICTIONS:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.gold_model_errors AS
    WITH e AS (
      SELECT
        l.city, l.city_name, l.city_role, l.id, l.name, l.listing_url, l.neighbourhood,
        l.room_type, l.property_type, l.stay_type, l.accommodates,
        p.split,
        p.split IN ('test', 'new_data')                      AS is_evaluation,
        {MODEL_RUN}                                          AS model_run,
        l.price                                              AS actual_price,
        round(p.predicted_price, 2)                          AS predicted_price,
        round(p.baseline_price, 2)                           AS baseline_price,
        round(p.predicted_price - l.price, 2)                AS error,
        round(abs(p.predicted_price - l.price), 2)           AS abs_error,
        round(abs(p.baseline_price - l.price), 2)            AS baseline_abs_error,
        round(100 * (p.predicted_price - l.price) / l.price, 1) AS error_pct,
        CASE WHEN p.predicted_price < l.price THEN 'under-priced' ELSE 'over-priced' END AS pricing
      FROM {PREDICTIONS} p
      JOIN {S}.gold_listings l ON p.city = l.city AND p.id = l.id
      WHERE l.price IS NOT NULL
    )
    -- latest run per city: price_model_v2 sorts after price_model_v1
    SELECT *, model_run = max(model_run) OVER (PARTITION BY city) AS is_latest_run
    FROM e
    """)

    metrics = """
      count(*)                                                    AS n_listings,
      round(avg(abs_error), 2)                                    AS model_mae,
      round(avg(baseline_abs_error), 2)                           AS baseline_mae,
      round(100 * (1 - avg(abs_error) / avg(baseline_abs_error)), 1) AS pct_better_than_baseline,
      round(median(abs_error), 2)                                 AS model_median_abs_error
    """
    spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.gold_model_metrics AS
    SELECT model_run, split, is_evaluation, city_name AS scope, city_role, {metrics}
    FROM {S}.gold_model_errors
    GROUP BY model_run, split, is_evaluation, city_name, city_role
    UNION ALL
    SELECT model_run, split, is_evaluation, 'A + B combined' AS scope, 'A+B' AS city_role, {metrics}
    FROM {S}.gold_model_errors
    WHERE city_role IN ('A', 'B')
    GROUP BY model_run, split, is_evaluation
    """)
    display(spark.table(f"{S}.gold_model_metrics").filter("is_evaluation").orderBy("model_run", "city_role"))

## 6. Table and column comments
Genie reads Unity Catalog comments, so this is where most of its "knowledge" about the columns comes from. Short and factual on purpose.

In [ ]:
COMMENTS = {
    "gold_listings": ("One row per Airbnb listing on the snapshot date, all cities.", {
        "city_role": "A, B or C. City A = Los Angeles, City B = San Francisco, City C = Santa Clara County (the new client).",
        "price": "Advertised price per night in USD. NULL when the listing has no price or a placeholder price, so it is safe to aggregate.",
        "stay_type": "short (minimum_nights < 28) or monthly (minimum_nights >= 28). Monthly rentals are priced about 40 % lower per night.",
        "price_class": "normal, low, high_explained (large home), high_unexplained, placeholder (fake price), no_price.",
        "is_stale": "True when Inside Airbnb carried the row over from an earlier scrape; these are mostly inactive.",
        "availability_365": "Days free to book in the next 365 days. 0 means fully booked or blocked.",
        "review_scores_rating": "Average guest rating, 0 to 5. NULL when the listing has no reviews.",
    }),
    "gold_city_summary": ("One row per city: size, prices by stay type, availability and reviews.", {
        "median_price_short_stay": "Median price per night in USD of short-stay listings with a valid price.",
        "median_price_monthly_stay": "Median price per night in USD of monthly rentals with a valid price.",
    }),
    "gold_neighbourhood_stats": ("One row per city and neighbourhood. Neighbourhood names only make sense within one city.", {
        "listings": "All listings in the neighbourhood, with or without a price.",
        "median_price_short_stay": "Median price per night in USD of short-stay listings with a valid price.",
    }),
    "gold_model_errors": ("One row per listing and model run, only listings with a valid actual price.", {
        "split": "train, test (held out, cities A and B) or new_data (city C, never used for training).",
        "is_evaluation": "True for rows whose errors are reported: test rows of A and B and all rows of city C. Filter on this.",
        "model_run": "Model version: price_model_v1 = end of week 1; a later version (price_model_v2) = after the city C changes.",
        "is_latest_run": "True for the most recent model run of that city.",
        "error": "predicted_price - actual_price in USD. Negative = the model under-prices the listing.",
        "abs_error": "Absolute error in USD; its mean is the MAE.",
        "baseline_price": "Median training price for the same room_type and accommodates.",
    }),
    "gold_model_metrics": ("Mean absolute error (MAE) of the model and the baseline per city, model run and split.", {
        "scope": "City name, or A + B combined for the two training cities together.",
        "is_evaluation": "True for the rows to report (test split of A and B, new_data of city C).",
        "pct_better_than_baseline": "How much lower the model MAE is than the baseline MAE, in %.",
    }),
}

for table, (table_comment, cols) in COMMENTS.items():
    if not spark.catalog.tableExists(f"{S}.{table}"):
        continue
    spark.sql(f"COMMENT ON TABLE {S}.{table} IS '{table_comment}'")
    for col, text in cols.items():
        spark.sql(f"ALTER TABLE {S}.{table} ALTER COLUMN {col} COMMENT '{text}'")

## 7. Checks

In [ ]:
# CHECK
n_silver, n_gold = spark.table(SILVER).count(), spark.table(f"{S}.gold_listings").count()
assert n_gold == n_silver, f"gold_listings has {n_gold:,} rows, the listings table has {n_silver:,}"
assert spark.table(f"{S}.gold_listings").filter("city_role IS NULL").count() == 0, "a city has no role in CITY_ROLES"
if HAS_PREDICTIONS:
    dup = spark.sql(f"SELECT city, id, model_run, count(*) n FROM {S}.gold_model_errors GROUP BY ALL HAVING n > 1").count()
    assert dup == 0, "gold_model_errors has more than one row per listing and model run"
print(f"gold tables in {S} are ready; listings: {n_gold:,}")